In [1]:
import torch
import json
import os
import logging
from datetime import datetime
from typing import List, Dict, Any, Optional
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
import warnings
warnings.filterwarnings('ignore')
import re
# Setup logging
logging.basicConfig(level=logging.INFO)
logger = logging.getLogger(__name__)

In [2]:
# Cell 2: Device setup and configuration
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"🔧 Using device: {device}")

# Main configuration
CONFIG = {
    "model_id": "meta-llama/Meta-Llama-3-8B-Instruct",  # Use Instruct version
    "use_quantization": True,
    "device": device,
    "max_new_tokens": 300,
    "temperature": 0.3,
    "input_file": "output_entities2.json",
    "output_dir": "lulc_extraction_output",
    "max_sentences": 20,  # Process first 50 sentences for testing
}

# Valid LULC relations
VALID_RELATIONS = [
    "CHANGE_TO", "INCREASES_BY", "DECREASES_BY", "CAUSES", "LOCATED_IN",
    "OCCURS_DURING", "MEASURES", "AFFECTS", "FROM_TO", "ENABLES"
]

# Create output directory
os.makedirs(CONFIG["output_dir"], exist_ok=True)
print(f" Configuration loaded successfully")
print(f" Output directory: {CONFIG['output_dir']}")
print(f" Model: {CONFIG['model_id']}")

🔧 Using device: cuda
 Configuration loaded successfully
 Output directory: lulc_extraction_output
 Model: meta-llama/Meta-Llama-3-8B-Instruct


In [3]:
# Cell 3: Data loading function
def load_preprocessed_data(file_path: str) -> List[Dict]:
    """Load already processed data with sentence and entities keys"""
    try:
        with open(file_path, 'r', encoding='utf-8') as f:
            data = json.load(f)
        
        print(f" Loaded {len(data)} items from {file_path}")
        
        processed_data = []
        for item in data:
            # Clean the sentence text
            sentence = item.get('sentence', '')
            if sentence.startswith("text': '"):
                sentence = sentence[7:]  # Remove "text': '"
            if sentence.endswith("'"):
                sentence = sentence[:-1]  # Remove trailing quote
            
            # Get existing entities (if any)
            entities = item.get('entities', [])
            
            processed_data.append({
                'sentence': sentence,
                'entities': entities,
                'original_data': item
            })
        
        # Print statistics
        total_entities = sum(len(item['entities']) for item in processed_data)
        sentences_with_entities = sum(1 for item in processed_data if item['entities'])
        
        print(f"📊 Processing Statistics:")
        print(f"  - Total sentences: {len(processed_data)}")
        print(f"  - Sentences with entities: {sentences_with_entities}")
        print(f"  - Total entities extracted: {total_entities}")
        if len(processed_data) > 0:
            print(f"  - Average entities per sentence: {total_entities/len(processed_data):.2f}")
        
        return processed_data
        
    except Exception as e:
        logger.error(f"Error loading data: {e}")
        return []

In [4]:
# Cell 4: Model loading function
def load_llama_model(model_id: str, use_quantization: bool = True):
    """Load Llama model with proper configuration"""
    print(f" Loading model: {model_id}")
    
    try:
        # Load tokenizer
        tokenizer = AutoTokenizer.from_pretrained(model_id)
        if tokenizer.pad_token is None:
            tokenizer.pad_token = tokenizer.eos_token
        
        print(f"📝 Tokenizer loaded. Vocab size: {tokenizer.vocab_size}")
        
        # Configure quantization for memory efficiency
        quantization_config = None
        if use_quantization and device.type == "cuda":
            quantization_config = BitsAndBytesConfig(
                load_in_4bit=True,
                bnb_4bit_compute_dtype=torch.float16,
                bnb_4bit_quant_type="nf4",
                bnb_4bit_use_double_quant=True
            )
            print("🔧 Using 4-bit quantization")
        
        # Load model
        model = AutoModelForCausalLM.from_pretrained(
            model_id,
            device_map="auto" if device.type == "cuda" else None,
            quantization_config=quantization_config,
            torch_dtype=torch.float16 if device.type == "cuda" else torch.float32,
            trust_remote_code=True
        )
        
        print(f"✅ Model loaded successfully on {model.device}")
        return model, tokenizer
        
    except Exception as e:
        logger.error(f"Failed to load model: {e}")
        raise

print(" Model loading function defined")

 Model loading function defined


In [5]:
def build_lulc_extraction_prompt(sentence):
    """Build prompt for joint entity recognition and relation extraction"""
    
    system_message = """You are an expert in Land Use Land Cover (LULC) analysis. Perform joint entity recognition and relation extraction from the given sentence.

**STRICT Entity Types (USE ONLY THESE):**
CHANGE: Any verb or noun phrase that denotes a measurable difference in land cover extent, quality, or characteristic between two or more temporal observations. CHANGE entities capture both the direction (increase, decrease, loss, gain) and nature (converted, degraded, improvement) of landscape dynamics
LOC: Any proper noun referring to a geographically identifiable area at any spatial scale, including continents, countries, regions, provinces, cities, villages, oror named places where LULC phenomena are observed.
LULC:Any noun or noun phrase that describes a specific type of land surface characterized by specific biophysical attributes or human utilization patterns. These entities encompass both natural land cover types (e.g., forests, woodlands, savannas) and anthropogenic land uses (e.g., agricultural land, cropped fields, fallows)
DATE: Any numeric expression or textual reference that specifies a particular year, decade, or other precise temporal point at which a land cover observation was made or a change , process occurred (1990, 2020, January 2000)
PERCENT:Any numerical expression followed by the percent symbol (%) or textual equivalent (e.g., "half") that represents a proportional measurement relative to a defined whole or baseline.
QUANTITY: Any numerical value that represents a count, ratio, index, or other non-percentage quantitative measure relevant to land use/land cover analysis.
COORDINATES: Any numerical expression representing geographic coordinates in a recognized coordinate reference system that specifies an exact location on Earth's surface. (40.7°N, latitude 23.5)
SURFACE_UNIT: Any expression that combines a numerical value with a standard unit of area measurement (e.g., km², hectares, acres)
PROCESS: Any noun or verb phrase that describes a natural phenomenon (e.g., droughts, desertification) and anthropogenic activities (e.g., cultivation, grazing, irrigation), human activity, or socio-ecological interaction that directly or indirectly causes, influences, or results from changes in land use/land cover patterns.
CARDINAL : pure numerical values without specific units or context
TIME_PERIODS : Time spans with beginning and end (1990-2000, between 1995 and 2005, from 2010 to 2020)


**ENTITY EXTRACTION RULES:**
1. Extract entities as concise as possible (e.g., "loss" not "observed loss of woody vegetation")
2. List each date separately (e.g., "1970s" and "1980s" as two entities)
3. "desertification" is a PROCESS (the process of becoming desert), not LULC
4. "woody vegetation" or "woody vegetation cover" is LULC
**FEW-SHOT EXAMPLES:**

**Relation Types:**

**CHANGE_TO**: Indicates a direct transformation from one LULC type to another.Source of the relation is LULC target of the relation is LULC 
-  CORRECT: forest --CHANGE_TO-- cropland (trees cut, land converted to farming)
-  CORRECT: agricultural land --CHANGE_TO-- urban area (farmland developed into city)
-  WRONG: built-up area --CHANGE_TO-- built-up area (same type, just quantity change)
-  WRONG: forest --CHANGE_TO-- forest (same type, just area change)
-  WRONG:the CHANGE_TO must be between 2 diffrence not same lulc 
**INCREASES_BY/DECREASES_BY**:Shows the magnitude of change (increase or decrease) in a land cover type, typically expressed as a percentage Source is lulc target is PERCENT or QUANTITY  
-  CORRECT: built-up area --INCREASES_BY-- 12.77% (more built-up area, not transformation)
-  CORRECT: forest --DECREASES_BY-- 25% (less forest area, not transformation)

**Other Relations:**
- CAUSES:Establishes a causal link between a process and a change. The process is the antecedent, and the change is the result. source is process target is change  (deforestation --CAUSES-- forest loss)
- LOCATED_IN: Specifies the spatial context by indicating the location where a land cover type, process, or other entity is found. source is lulc process , change target is loc  (forest --LOCATED_IN-- Brazil)
- OCCURS_DURING: Indicates the temporal context by specifying when a process or change took place. It can link to a specific period of time. source is PROCESS, CHANGE target is TIME_PERIOD(change --OCCURS_DURING-- 2018)
- MEASURES: This is a relation to quantify, it means that a quantity that serves as measure or description for some entity. source is PERCENT, CARDINAL,QUANTITY, SURFACE_UNIT target is change , lulc (12.77% --MEASURES-- increase)
- AFFECTS: Describes the impact of a process or change on LULC. must be a real impact (urbanization --AFFECTS-- forest)
- FROM_TO: Value changes between 2 persentage entity or 2 SURFACE_UNIT entity (52.88% --FROM_TO-- 65.5%)
- ENABLES: One process enables another process (deforestation --ENABLES-- urbanization)

**CRITICAL THINKING RULES:**
1. Ask yourself: Is this ACTUALLY a transformation between different land types?
2. Think about the process: What physical change happened to the land?
3. Consider causality: What caused what? Don't create meaningless loops
4. Be precise with measurements: Percentages usually MEASURE changes, not cause them
5. Temporal logic: Changes happen DURING time periods, not TO time periods
6. Spatial logic: Things are LOCATED_IN places, places don't transform to places

INSTRUCTIONS:
1. Extract ONLY relations that are explicitly stated or directly implied in the sentence
2. Use ONLY the entities provided above
3. Each relation must include the entity label in the format: entity_text:ENTITY_LABEL
4. Each relation must follow the format: source_entity:SOURCE_LABEL --RELATIONSHIP-- target_entity:TARGET_LABEL
5. Include confidence level (HIGH/MEDIUM/LOW) for each relation
6. Do not create relations between entities of the same type using CHANGE_TO
Let's think step by step. First, identify all the entities in the sentence based on the provided Entity Types. Then, determine the relationships between those entities according to the Relation Types, considering the Critical Thinking Rules. Finally, format the output as specified.
**OUTPUT FORMAT:**
ENTITIES:
- entity_text | ENTITY_TYPE

RELATIONS:
- entity:ENTITY_TYPE --RELATIONSHIP-- entity:ENTITY_TYPE | CONF: confidence_level

Extract only what is explicitly mentioned. Be concise and accurate."""

    # Llama 3 format with special tokens
    prompt = f"""<|begin_of_text|><|start_header_id|>system<|end_header_id|>
{system_message}<|eot_id|>
<|start_header_id|>user<|end_header_id|>
Analyze this sentence: "{sentence}"<|eot_id|>
<|start_header_id|>assistant<|end_header_id|>
ENTITIES:
-"""
    
    return prompt
print(" Prompt engineering function defined for Llama 3")

 Prompt engineering function defined for Llama 3


In [6]:
def generate_lulc_extraction(sentence, model, tokenizer):
    """Generate entity and relation extraction using Llama 3"""
    prompt = build_lulc_extraction_prompt(sentence)
    
    # Tokenize with proper settings for Llama 3
    inputs = tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True,  # Enable truncation for safety
        max_length=4096,    # Llama 3 8B context window
        padding=True,
        return_attention_mask=True
    )
    
    # Move to device
    inputs = {k: v.to(model.device) for k, v in inputs.items()}
    
    # Generate with Llama 3 optimized settings
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=CONFIG["max_new_tokens"],
            temperature=CONFIG["temperature"],
            do_sample=True,
            top_p=0.9,
            top_k=50,
            repetition_penalty=1.1,
            pad_token_id=tokenizer.pad_token_id,
            eos_token_id=tokenizer.eos_token_id,
            use_cache=True
        )
    
    # Decode and return
    response = tokenizer.decode(
        outputs[0][inputs['input_ids'].shape[1]:],
        skip_special_tokens=True
    )
    
    return response

print(" Generation function defined")

 Generation function defined


In [7]:
def clean_and_parse_response(response: str, original_sentence: str) -> Dict[str, Any]:
    """Improved parsing that handles both colon and pipe separators"""
    
    result = {
        'entities': [],
        'relations': [],
        'raw_response': response
    }
    
    if not response:
        return result
    
    lines = response.split('\n')
    current_section = "ENTITIES"
    
    for line in lines:
        line = line.strip()
        if not line:
            continue
            
        # Check for section transitions
        if line.upper().startswith("RELATIONS"):
            current_section = "RELATIONS"
            continue
            
        # Parse entities - handles both formats
        if current_section == "ENTITIES":
            # Remove leading dashes/bullets
            clean_line = line.lstrip('-•· ').strip()
            
            # Try colon format: "text:TYPE"
            if ':' in clean_line and not '|' in clean_line:
                parts = clean_line.split(':', 1)
                text = parts[0].strip()
                entity_type = parts[1].strip()
                result['entities'].append({'text': text, 'type': entity_type})
                
            # Try pipe format: "text | TYPE"
            elif '|' in clean_line:
                parts = clean_line.split('|', 1)
                text = parts[0].strip()
                entity_type = parts[1].strip()
                result['entities'].append({'text': text, 'type': entity_type})
                
            # Fallback: Check if line contains a valid entity type
            else:
                valid_types = ['CHANGE', 'LOC', 'LULC', 'DATE', 'PERCENT', 'CARDINAL', 
                              'COORDINATES', 'SURFACE_UNIT', 'PROCESS', 'QUANTITY']
                for etype in valid_types:
                    if etype in line:
                        text = line.replace(etype, '').strip(' :-|')
                        if text:
                            result['entities'].append({'text': text, 'type': etype})
                            
        # Parse relations
        elif current_section == "RELATIONS":
            if "--" in line:
                relation_text = line.lstrip('-•· ').strip()
                result['relations'].append(relation_text)
    
    # Extract missing PERCENT entities
    percentage_pattern = r'\b\d+\.?\d*%\b'
    for relation in result['relations']:
        percentages = re.findall(percentage_pattern, relation)
        for pct in percentages:
            pct_exists = any(entity['text'] == pct for entity in result['entities'])
            if not pct_exists and pct in original_sentence:
                result['entities'].append({'text': pct, 'type': 'PERCENT'})
    
    # Remove duplicates
    seen = set()
    unique_entities = []
    for entity in result['entities']:
        ident = (entity['text'].lower(), entity['type'])
        if ident not in seen:
            seen.add(ident)
            unique_entities.append(entity)
            
    result['entities'] = unique_entities
    return result

# Test with your model output
test_response = """ After the droughts in the 1970s and 1980s | DATE
- loss of woody vegetation cover | CHANGE
- woody vegetation cover | LULC
- Sahel | LOC
- degraded land | LULC

RELATIONS:
- droughts in the 1970s and 1980s:DATE --CAUSES-- loss of woody vegetation cover:CHANGE | HIGH
- loss of woody vegetation cover:CHANGE --AFFECTS-- woody vegetation cover:LULC | HIGH
- Sahel:LOC --LOCATED_IN-- loss of woody vegetation cover:CHANGE | HIGH
- loss of woody vegetation cover:CHANGE --LEADS_TO-- degraded land:LULC | HIGH"""

test_sentence = "After the droughts in the 1970s and 1980s, the observed loss of woody vegetation cover was often considered as irreversible and large parts of the Sahel"

parsed = clean_and_parse_response(test_response, test_sentence)
print(" Parsing Results:")
print(f"Entities: {len(parsed['entities'])}")
for e in parsed['entities']:
    print(f" - '{e['text']}' | {e['type']}")
print(f"\nRelations: {len(parsed['relations'])}")
for r in parsed['relations']:
    print(f" - {r}")

 Parsing Results:
Entities: 5
 - 'After the droughts in the 1970s and 1980s' | DATE
 - 'loss of woody vegetation cover' | CHANGE
 - 'woody vegetation cover' | LULC
 - 'Sahel' | LOC
 - 'degraded land' | LULC

Relations: 4
 - droughts in the 1970s and 1980s:DATE --CAUSES-- loss of woody vegetation cover:CHANGE | HIGH
 - loss of woody vegetation cover:CHANGE --AFFECTS-- woody vegetation cover:LULC | HIGH
 - Sahel:LOC --LOCATED_IN-- loss of woody vegetation cover:CHANGE | HIGH
 - loss of woody vegetation cover:CHANGE --LEADS_TO-- degraded land:LULC | HIGH


In [8]:
def process_sentences_batch(sentences: List[Dict], model, tokenizer, max_sentences: int = None) -> List[Dict]:
    """Process multiple sentences and extract LULC information"""
    
    if max_sentences:
        sentences = sentences[:max_sentences]
    
    results = []
    
    print(f" Processing {len(sentences)} sentences...")
    
    for i, item in enumerate(sentences, 1):
        sentence = item['sentence']
        
        if len(sentence) < 10:  # Skip very short sentences
            continue
            
        print(f"📝 Processing {i}/{len(sentences)}: {sentence[:60]}...")
        
        try:
            # Generate extraction
            raw_response = generate_lulc_extraction(sentence, model, tokenizer)
            
            # Parse and clean
            parsed_result = clean_and_parse_response(raw_response, sentence)
            
            # Combine with original data
            result = {
                'sentence': sentence,
                'original_entities': item.get('entities', []),
                'extracted_entities': parsed_result['entities'],
                'extracted_relations': parsed_result['relations'],
                'raw_model_response': parsed_result['raw_response'],
                'processing_timestamp': datetime.now().isoformat()
            }
            
            results.append(result)
            
            # Print progress
            if parsed_result['entities']:
                print(f"    Found {len(parsed_result['entities'])} entities, {len(parsed_result['relations'])} relations")
            else:
                print(f"    No entities found")
                
        except Exception as e:
            logger.error(f"Error processing sentence {i}: {e}")
            result = {
                'sentence': sentence,
                'original_entities': item.get('entities', []),
                'extracted_entities': [],
                'extracted_relations': [],
                'error': str(e),
                'processing_timestamp': datetime.now().isoformat()
            }
            results.append(result)
            continue
            
        # Save intermediate results every 10 sentences
        if i % 10 == 0:
            save_results(results, f"{CONFIG['output_dir']}/intermediate_results_{i}.json")
    
    return results

print("✅ Batch processing function defined")

# Cell 9: Save results function
def save_results(results: List[Dict], output_path: str):
    """Save processing results to JSON file"""
    try:
        with open(output_path, 'w', encoding='utf-8') as f:
            json.dump(results, f, indent=2, ensure_ascii=False)
        print(f" Saved {len(results)} results to {output_path}")
    except Exception as e:
        logger.error(f"Error saving results: {e}")


# Load the model
print("\n🚀 Starting LULC extraction pipeline...")
model, tokenizer = load_llama_model(CONFIG["model_id"], CONFIG["use_quantization"])

# Load input data
print("\n Loading input data...")
input_data = load_preprocessed_data(CONFIG["input_file"])


✅ Batch processing function defined

🚀 Starting LULC extraction pipeline...
 Loading model: meta-llama/Meta-Llama-3-8B-Instruct
📝 Tokenizer loaded. Vocab size: 128000
🔧 Using 4-bit quantization


INFO:accelerate.utils.modeling:We will use 90% of the memory on device 0 for storing the model, and 10% for the buffer to avoid OOM. You can set `max_memory` in to a higher value to use more memory (at your own risk).


Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

✅ Model loaded successfully on cuda:0

 Loading input data...
 Loaded 67 items from output_entities2.json
📊 Processing Statistics:
  - Total sentences: 67
  - Sentences with entities: 67
  - Total entities extracted: 398
  - Average entities per sentence: 5.94


In [9]:

# Test with single sentence
if input_data:
    print("\n Testing with single sentence...")
    test_sentence = input_data[0]['sentence']
    print(f"Test sentence: {test_sentence[:150]}...")
    
    # Generate response
    test_response = generate_lulc_extraction(test_sentence, model, tokenizer)
    print(f"\n Raw model response:")
    print(f"'{test_response}'")
    
    # Parse response
    test_parsed = clean_and_parse_response(test_response, test_sentence)
    
    print(f"\n Parsed Results:")
    print(f"   - Entities found: {len(test_parsed['entities'])}")
    for entity in test_parsed['entities']:
        print(f"     • {entity['text']} | {entity['type']}")
    
    print(f"   - Relations found: {len(test_parsed['relations'])}")
    for relation in test_parsed['relations']:
        print(f"     • {relation}")
else:
    print(" No data available for testing")


 Testing with single sentence...
Test sentence: #text: After the droughts in the 1970s and 1980s, the observed loss of woody vegetation cover was often considered as irreversible desertification and...

 Raw model response:
' droughts | PROCESS
- 1970s | DATE
- 1980s | DATE
- loss of woody vegetation cover | CHANGE
- irreversible desertification | PROCESS
- Sahel | LOC
- degraded land | LULC

RELATIONS:
- droughts:PROCESS --CAUSES-- loss of woody vegetation cover:CHANGE | CONF: HIGH
- loss of woody vegetation cover:CHANGE --IS_CONSIDERED_AS-- irreversible desertification:PROCESS | CONF: MEDIUM
- irreversible desertification:PROCESS --RESULTS_IN-- degraded land:LULC | CONF: HIGH'

 Parsed Results:
   - Entities found: 7
     • droughts | PROCESS
     • 1970s | DATE
     • 1980s | DATE
     • loss of woody vegetation cover | CHANGE
     • irreversible desertification | PROCESS
     • Sahel | LOC
     • degraded land | LULC
   - Relations found: 3
     • droughts:PROCESS --CAUSES-- loss 

In [10]:
if input_data:
    print(f"\n Processing {min(len(input_data), CONFIG['max_sentences'])} sentences...")
    results = process_sentences_batch(
        input_data, 
        model, 
        tokenizer, 
        max_sentences=CONFIG["max_sentences"]
    )
    
    # Save final results
    final_output_path = f"{CONFIG['output_dir']}/llama3_lulc_extraction_results with cot.json"
    save_results(results, final_output_path)
    
    # Print summary statistics
    print("\n Final Statistics:")
    print(f"  - Total sentences processed: {len(results)}")
    
    sentences_with_entities = sum(1 for r in results if r['extracted_entities'])
    sentences_with_relations = sum(1 for r in results if r['extracted_relations'])
    total_entities = sum(len(r['extracted_entities']) for r in results)
    total_relations = sum(len(r['extracted_relations']) for r in results)
    
    print(f"  - Sentences with entities: {sentences_with_entities} ({sentences_with_entities/len(results)*100:.1f}%)")
    print(f"  - Sentences with relations: {sentences_with_relations} ({sentences_with_relations/len(results)*100:.1f}%)")
    print(f"  - Total entities extracted: {total_entities}")
    print(f"  - Total relations extracted: {total_relations}")
    print(f"  - Average entities per sentence: {total_entities/len(results):.2f}")
    print(f"  - Average relations per sentence: {total_relations/len(results):.2f}")

# Cell 13: Analyze results
def analyze_extraction_quality(results: List[Dict]):
    """Analyze the quality of extractions"""
    entity_types = {}
    relation_types = {}
    
    for result in results:
        for entity in result['extracted_entities']:
            entity_type = entity['type']
            entity_types[entity_type] = entity_types.get(entity_type, 0) + 1
        
        for relation in result['extracted_relations']:
            # Extract relation type
            if "--" in relation:
                parts = relation.split("--")
                if len(parts) >= 2:
                    rel_type = parts[1].split("--")[0].strip()
                    relation_types[rel_type] = relation_types.get(rel_type, 0) + 1
    
    print("\n Entity Type Distribution:")
    for entity_type, count in sorted(entity_types.items(), key=lambda x: x[1], reverse=True):
        print(f"  - {entity_type}: {count}")
    
    print("\n Relation Type Distribution:")
    for rel_type, count in sorted(relation_types.items(), key=lambda x: x[1], reverse=True):
        print(f"  - {rel_type}: {count}")

if 'results' in locals():
    analyze_extraction_quality(results)

print("\n LULC extraction pipeline completed successfully!")


 Processing 20 sentences...
 Processing 20 sentences...
📝 Processing 1/20: #text: After the droughts in the 1970s and 1980s, the observ...
    Found 7 entities, 4 relations
📝 Processing 2/20: #text: The forests of West and Central Africa probably origi...
    Found 3 entities, 0 relations
📝 Processing 3/20: )., head: Land use change in the study site, p: ref: 2a, 2b,...
    Found 12 entities, 4 relations
📝 Processing 4/20: Agriculture in this region has been dominated over the past ...
    Found 3 entities, 1 relations
📝 Processing 5/20: Forest land was decreased nearly by half in 2002 compared to...
    Found 4 entities, 5 relations
📝 Processing 6/20: About 26 and 31% of the total area of natural vegetation (fo...
    Found 13 entities, 9 relations
📝 Processing 7/20: Assuming the dynamics recorded in the second period , the am...
    Found 10 entities, 5 relations
📝 Processing 8/20: Conversely the decrease in the herbaceous standing crop, due...
    Found 7 entities, 7 relations
📝 Pr

In [11]:
import csv
import re
from typing import List, Dict, Tuple, Optional

def parse_relation(relation_str: str) -> Optional[Tuple[str, str, str, str, str, str]]:
    """
    Parse a relation string to extract source, source_type, relationship, target, target_type, and confidence
    
    Expected format: source:SOURCE_TYPE --RELATIONSHIP-- target:TARGET_TYPE | CONF: confidence
    """
    try:
        # Initialize confidence as empty
        confidence = ""
        
        # Extract confidence if present
        if "| CONF:" in relation_str:
            parts = relation_str.split("| CONF:")
            relation_str = parts[0].strip()
            confidence = parts[1].strip()
        elif "|" in relation_str and relation_str.endswith(("HIGH", "MEDIUM", "LOW")):
            # Handle format: ... | HIGH
            parts = relation_str.rsplit("|", 1)
            relation_str = parts[0].strip()
            confidence = parts[1].strip()
        
        # Split by the relationship marker
        if "--" not in relation_str:
            return None
            
        # Find the relationship type (between --)
        parts = relation_str.split("--")
        if len(parts) < 3:
            return None
            
        # Extract components
        source_part = parts[0].strip()
        relationship = parts[1].strip()
        target_part = parts[2].strip()
        
        # Parse source (format: text:TYPE)
        if ":" not in source_part:
            return None
        source_split = source_part.rsplit(":", 1)
        source = source_split[0].strip()
        source_type = source_split[1].strip()
        
        # Parse target (format: text:TYPE)
        if ":" not in target_part:
            return None
        target_split = target_part.rsplit(":", 1)
        target = target_split[0].strip()
        target_type = target_split[1].strip()
        
        return source, source_type, relationship, target, target_type, confidence
        
    except Exception as e:
        print(f"Error parsing relation: {relation_str} - {e}")
        return None

def save_results_to_csv(results: List[Dict], output_path: str):
    """
    Save extraction results to CSV with proper relation parsing including confidence
    """
    
    # Prepare CSV data
    csv_rows = []
    sentence_id = 1
    
    print(f"\n📊 Processing results for CSV export...")
    
    for result in results:
        sentence = result['sentence']
        relations = result.get('extracted_relations', [])
        
        if not relations:
            # Add row even if no relations found
            csv_rows.append({
                'sentenceID': sentence_id,
                'sentence': sentence,
                'source': '',
                'source_type': '',
                'relationship': '',
                'target': '',
                'target_type': '',
                'confidence': ''
            })
        else:
            # Process each relation
            valid_relations = 0
            for relation in relations:
                parsed = parse_relation(relation)
                if parsed:
                    source, source_type, relationship, target, target_type, confidence = parsed
                    csv_rows.append({
                        'sentenceID': sentence_id,
                        'sentence': sentence,
                        'source': source,
                        'source_type': source_type,
                        'relationship': relationship,
                        'target': target,
                        'target_type': target_type,
                        'confidence': confidence
                    })
                    valid_relations += 1
            
            if valid_relations > 0:
                print(f"  Sentence {sentence_id}: {valid_relations}/{len(relations)} valid relations")
        
        sentence_id += 1
    
    # Write to CSV
    if csv_rows:
        with open(output_path, 'w', newline='', encoding='utf-8') as csvfile:
            fieldnames = ['sentenceID', 'sentence', 'source', 'source_type', 'relationship', 
                         'target', 'target_type', 'confidence']
            writer = csv.DictWriter(csvfile, fieldnames=fieldnames)
            
            writer.writeheader()
            writer.writerows(csv_rows)
        
        print(f"\n✅ CSV file saved to: {output_path}")
        print(f"   Total rows: {len(csv_rows)}")
        
        # Print statistics
        sentences_with_relations = len(set(row['sentenceID'] for row in csv_rows if row['source']))
        total_relations = sum(1 for row in csv_rows if row['source'])
        
        print(f"\n📈 CSV Statistics:")
        print(f"   - Total sentences: {sentence_id - 1}")
        print(f"   - Sentences with valid relations: {sentences_with_relations}")
        print(f"   - Total valid relations: {total_relations}")
        
        # Show sample of the CSV content
        print(f"\n📋 Sample CSV content (first 5 relations):")
        sample_count = 0
        for row in csv_rows:
            if row['source'] and sample_count < 5:
                print(f"   {row['sentenceID']} | {row['source']}:{row['source_type']} "
                      f"--{row['relationship']}-- {row['target']}:{row['target_type']} "
                      f"| CONF: {row['confidence']}")
                sample_count += 1
    else:
        print("⚠️ No data to save to CSV")

# Execute the CSV export with confidence column
if 'results' in locals() and results:
    csv_output_path = f"{CONFIG['output_dir']}/llama3_lulc_relations_with_cot.csv"
    save_results_to_csv(results, csv_output_path)
    
    # Test the parsing function with sample relations
    print("\n🔍 Testing relation parsing with confidence...")
    test_relations = [
        "forest:LULC --CHANGE_TO-- cropland:LULC | HIGH",
        "deforestation:PROCESS --CAUSES-- forest:LULC | CONF: MEDIUM",
        "urban area:LULC --INCREASES_BY-- 25%:PERCENT | LOW"
    ]
    
    for rel in test_relations:
        parsed = parse_relation(rel)
        if parsed:
            s, st, r, t, tt, c = parsed
            print(f"✅ Parsed: {s} ({st}) --{r}-- {t} ({tt}) | Confidence: {c}")
else:
    print("❌ No results found to export to CSV")


📊 Processing results for CSV export...
  Sentence 1: 4/4 valid relations
  Sentence 3: 4/4 valid relations
  Sentence 4: 1/1 valid relations
  Sentence 5: 1/5 valid relations
  Sentence 6: 3/9 valid relations
  Sentence 9: 2/5 valid relations
  Sentence 10: 5/5 valid relations
  Sentence 11: 4/4 valid relations
  Sentence 13: 6/6 valid relations
  Sentence 14: 8/8 valid relations
  Sentence 15: 4/4 valid relations
  Sentence 16: 1/1 valid relations
  Sentence 17: 4/4 valid relations
  Sentence 18: 5/6 valid relations
  Sentence 19: 8/8 valid relations
  Sentence 20: 4/4 valid relations

✅ CSV file saved to: lulc_extraction_output/llama3_lulc_relations_with_cot.csv
   Total rows: 65

📈 CSV Statistics:
   - Total sentences: 20
   - Sentences with valid relations: 16
   - Total valid relations: 64

📋 Sample CSV content (first 5 relations):
   1 | droughts:PROCESS --CAUSES-- loss of woody vegetation cover:CHANGE | CONF: HIGH
   1 | loss of woody vegetation cover:CHANGE --MEASURES-- irreve